In [ ]:
# 1. Полное удаление всех следов NumPy
!pip uninstall -y numpy numpy-base

# 2. Очистка кэша pip
!pip cache purge

# 3. Установка стабильной, проверенной версии NumPy
!pip install --no-cache-dir numpy==1.26.4

print("✅ NumPy переустановлен.")
print("⚠️ ОЧЕНЬ ВАЖНО: ТЕПЕРЬ ОБЯЗАТЕЛЬНО ПЕРЕЗАПУСТИТЕ СРЕДУ ВЫПОЛНЕНИЯ!")
print("   Меню сверху: Среда выполнения -> Перезапустить среду выполнения")

Found existing installation: numpy 2.0.2
Uninstalling numpy-2.0.2:
  Successfully uninstalled numpy-2.0.2
Files removed: 0
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 299.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opencv-contrib-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
tobler 0.14.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
jaxlib 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
cupy-cuda12x 14.0.1 requires numpy<2.6,>=2.0, but you have numpy 1.26.4 which is incompatible.
opencv-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
tifffile 2026.4.11 requires numpy>=2.0

✅ NumPy переустановлен.
⚠️ ОЧЕНЬ ВАЖНО: ТЕПЕРЬ ОБЯЗАТЕЛЬНО ПЕРЕЗАПУСТИТЕ СРЕДУ ВЫПОЛНЕНИЯ!
   Меню сверху: Среда выполнения -> Перезапустить среду выполнения


In [ ]:
# Установка пакетов для сегментации и конвертации (без импортов!)
!pip install -q segmentation_models_pytorch transformers albumentations torchmetrics pytorch-lightning
!pip install -q onnx onnxruntime onnxslim tf2onnx onnx2tf tensorflowjs h5py onnxconverter-common

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 29.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 852.4/852.4 kB 28.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 43.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
numba 0.60.0 requires numpy<2.1,>=1.22, but you have numpy 2.4.6 which is incompatible.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 223.2/223.2 kB 6.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 853.3 kB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.7/57.7 kB 6.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.2/15.2 MB 96.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 238.4/23

In [ ]:
# Импорты (после установки)
import os, time, glob, json, torch, torch.nn as nn
import torchmetrics, onnx, onnxruntime, tensorflow as tf
from PIL import Image
import pytorch_lightning as pl
import albumentations as A
from albumentations.pytorch import ToTensorV2
from torch.utils.data import Dataset
from google.colab import drive
import matplotlib.pyplot as plt
import pandas as pd
from collections import defaultdict
from tqdm import tqdm

# Важно: удаляем scipy из памяти, чтобы избежать конфликта с устаревшим numpy
import sys
if 'scipy' in sys.modules:
    del sys.modules['scipy']

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
NUM_CLASSES = 8
IMG_SIZE = 512
CLASS_NAMES = ['Background', 'Building', 'Road', 'Static Car', 'Tree', 'Low Veg', 'Human', 'Moving Car']

print(f"✅ Устройство: {DEVICE}")
print(f"✅ TensorFlow версия: {tf.__version__}")
print("✅ Импорты выполнены (scipy временно удалён)")

✅ Устройство: cuda
✅ TensorFlow версия: 2.19.1
✅ Импорты выполнены (scipy временно удалён)


In [ ]:
drive.mount('/content/drive')

# Пути для чекпоинтов и моделей
DRIVE_CKPT_DIR = '/content/drive/MyDrive/uavid_lab/checkpoints'
DRIVE_MODELS_DIR = '/content/drive/MyDrive/uavid_lab/converted_models'
os.makedirs(DRIVE_CKPT_DIR, exist_ok=True)
os.makedirs(DRIVE_MODELS_DIR, exist_ok=True)

# Скачивание датасета (Kaggle)
import kagglehub
print("📥 Загрузка UAVid датасета...")
try:
    DATASET_ROOT = kagglehub.dataset_download("dasmehdixtr/uavid-v1")
    print(f"✅ Датасет загружен в: {DATASET_ROOT}")
except Exception as e:
    print("⚠️ Ошибка Kaggle. Загрузите архив вручную в Drive и укажите путь:")
    DATASET_ROOT = '/content/drive/MyDrive/uavid-v1'

Mounted at /content/drive
📥 Загрузка UAVid датасета...


100%|██████████| 10.9G/10.9G [02:19<00:00, 84.6MB/s]

Extracting files...


✅ Датасет загружен в: /root/.cache/kagglehub/datasets/dasmehdixtr/uavid-v1/versions/3


In [ ]:
import numpy as np
UAVID_COLORMAP = {
    (0, 0, 0): 0, (128, 0, 0): 1, (128, 64, 128): 2, (0, 0, 70): 3,
    (0, 128, 0): 4, (128, 128, 0): 5, (128, 0, 128): 6, (64, 64, 0): 7,
}

def rgb_to_mask(mask_rgb):
    h, w = mask_rgb.shape[:2]
    mask = np.zeros((h, w), dtype=np.int64)
    for color, class_id in UAVID_COLORMAP.items():
        mask[np.all(mask_rgb == np.array(color), axis=-1)] = class_id
    return mask

class UAVidDataset(Dataset):
    def __init__(self, root_dir, split='uavid_train', img_size=IMG_SIZE):
        self.root_dir = root_dir
        self.img_size = img_size
        self.image_paths, self.mask_paths = [], []

        split_dir = os.path.join(root_dir, split)
        if not os.path.exists(split_dir):
            raise FileNotFoundError(f"Split not found: {split_dir}")

        seq_dirs = sorted([d for d in os.listdir(split_dir) if os.path.isdir(os.path.join(split_dir, d))])
        for seq in seq_dirs:
            img_dir = os.path.join(split_dir, seq, 'Images')
            mask_dir = os.path.join(split_dir, seq, 'Labels')
            if os.path.exists(img_dir) and os.path.exists(mask_dir):
                img_files = sorted([f for f in os.listdir(img_dir) if f.lower().endswith(('.jpg','.png'))])
                mask_files = sorted([f for f in os.listdir(mask_dir) if f.lower().endswith(('.jpg','.png'))])
                for i_f, m_f in zip(img_files, mask_files):
                    self.image_paths.append(os.path.join(img_dir, i_f))
                    self.mask_paths.append(os.path.join(mask_dir, m_f))

        is_train = split == 'uavid_train'
        self.transform = A.Compose([
            A.RandomCrop(img_size, img_size) if is_train else A.Resize(img_size, img_size),
            A.HorizontalFlip(p=0.5) if is_train else A.NoOp(),
            A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
            ToTensorV2(),
        ], additional_targets={'mask': 'mask'})

    def __len__(self): return len(self.image_paths)

    def __getitem__(self, idx):
        image = np.array(Image.open(self.image_paths[idx]).convert("RGB"))
        mask_rgb = np.array(Image.open(self.mask_paths[idx]).convert("RGB"))
        transformed = self.transform(image=image, mask=rgb_to_mask(mask_rgb))
        return transformed["image"], transformed["mask"].long()

train_ds = UAVidDataset(DATASET_ROOT, 'uavid_train')
val_ds   = UAVidDataset(DATASET_ROOT, 'uavid_val')
train_loader = torch.utils.data.DataLoader(train_ds, batch_size=4, shuffle=True, num_workers=2, pin_memory=True)
val_loader   = torch.utils.data.DataLoader(val_ds, batch_size=4, shuffle=False, num_workers=2, pin_memory=True)
print(f"✅ Train: {len(train_ds)}, Val: {len(val_ds)}")

✅ Train: 600, Val: 70


In [ ]:
import segmentation_models_pytorch as smp

class SegModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = smp.Unet(encoder_name="resnet34", encoder_weights="imagenet", in_channels=3, classes=NUM_CLASSES)
    def forward(self, x): return self.model(x)

model = SegModel().to(DEVICE)
ckpt_path = os.path.join(DRIVE_CKPT_DIR, 'unet_resnet34_base.pth')
torch.save({
    'model_state_dict': model.state_dict(),
    'config': {'arch': 'unet', 'encoder': 'resnet34', 'classes': NUM_CLASSES}
}, ckpt_path)
print(f"💾 Чекпоинт сохранен в Drive: {ckpt_path}")

💾 Чекпоинт сохранен в Drive: /content/drive/MyDrive/uavid_lab/checkpoints/unet_resnet34_base.pth


In [ ]:
def plot_class_distribution(dataset, class_names, save_dir='./plots'):
    os.makedirs(save_dir, exist_ok=True)
    counts = defaultdict(int)
    for _, target in tqdm(dataset, desc='Анализ датасета', leave=False):
        for lbl in target['labels'].numpy():
            if lbl > 0: counts[lbl] += 1
    labels = [i for i in range(1, len(class_names)) if counts[i] > 0]
    values = [counts[i] for i in labels]
    names = [class_names[i] for i in labels]
    plt.figure(figsize=(12,5))
    plt.bar(names, values, color='steelblue')
    plt.xticks(rotation=45, ha='right')
    plt.title('Распределение классов в тестовом наборе SIMD')
    plt.tight_layout()
    plt.savefig(os.path.join(save_dir, 'class_distribution.png'), dpi=150, bbox_inches='tight')
    plt.show()

def plot_ap_per_class(aps_dict, class_names, save_dir='./plots'):
    os.makedirs(save_dir, exist_ok=True)
    class_ids = list(range(1, NUM_CLASSES))  # исправлено: CONFIG -> NUM_CLASSES
    x = np.arange(len(class_ids))
    width = 0.25
    models = list(aps_dict.keys())
    colors = {'YOLOv9':'#3498db', 'YOLO26':'#2ecc71', 'Faster R-CNN':'#e74c3c'}
    plt.figure(figsize=(14,6))
    for i, (name, aps) in enumerate(aps_dict.items()):
        vals = [aps.get(cid, 0) for cid in class_ids]
        plt.bar(x + (i-1)*width, vals, width, label=name, color=colors.get(name, 'gray'), alpha=0.85)
    plt.xticks(x, [class_names[i] for i in class_ids], rotation=45, ha='right')
    plt.ylabel('AP@50'); plt.ylim(0, 1.05); plt.title('AP@50 по классам')
    plt.legend(); plt.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(save_dir, 'ap_per_class.png'), dpi=150, bbox_inches='tight')
    plt.show()

def plot_comparison_table(results, save_dir='./plots'):
    os.makedirs(save_dir, exist_ok=True)
    def best3(v9, v26, vr, hi=True):
        vals = [(v9, 'YOLOv9'), (v26, 'YOLO26'), (vr, 'Faster R-CNN')]
        return (max if hi else min)(vals, key=lambda x: x[0])[1]

    rows = [
        ('mAP@50', f"{results['YOLOv9']['map']:.4f}", f"{results['YOLO26']['map']:.4f}", f"{results['Faster R-CNN']['map']:.4f}", best3(results['YOLOv9']['map'], results['YOLO26']['map'], results['Faster R-CNN']['map'])),
        ('Precision', f"{results['YOLOv9']['p']:.4f}", f"{results['YOLO26']['p']:.4f}", f"{results['Faster R-CNN']['p']:.4f}", best3(results['YOLOv9']['p'], results['YOLO26']['p'], results['Faster R-CNN']['p'])),
        ('Recall', f"{results['YOLOv9']['r']:.4f}", f"{results['YOLO26']['r']:.4f}", f"{results['Faster R-CNN']['r']:.4f}", best3(results['YOLOv9']['r'], results['YOLO26']['r'], results['Faster R-CNN']['r'])),
        ('F1-Score', f"{results['YOLOv9']['f1']:.4f}", f"{results['YOLO26']['f1']:.4f}", f"{results['Faster R-CNN']['f1']:.4f}", best3(results['YOLOv9']['f1'], results['YOLO26']['f1'], results['Faster R-CNN']['f1'])),
        ('Инференс (мс/кадр)', f"{results['YOLOv9']['ms']:.1f}", f"{results['YOLO26']['ms']:.1f}", f"{results['Faster R-CNN']['ms']:.1f}", best3(results['YOLOv9']['ms'], results['YOLO26']['ms'], results['Faster R-CNN']['ms'], hi=False)),
        ('FPS', f"{results['YOLOv9']['fps']:.1f}", f"{results['YOLO26']['fps']:.1f}", f"{results['Faster R-CNN']['fps']:.1f}", best3(results['YOLOv9']['fps'], results['YOLO26']['fps'], results['Faster R-CNN']['fps'])),
    ]
    fig, ax = plt.subplots(figsize=(13, 3.5))
    ax.axis('off')
    col_labels = ['Метрика', 'YOLOv9', 'YOLO26', 'Faster R-CNN', 'Победитель']
    table = ax.table(cellText=rows, colLabels=col_labels, loc='center', cellLoc='center')
    table.auto_set_font_size(False); table.set_fontsize(11); table.scale(1, 1.8)
    for j, color in enumerate(['#2c3e50', '#3498db', '#2ecc71', '#e74c3c', '#2c3e50']):
        table[0, j].set_facecolor(color); table[0, j].set_text_props(color='white', fontweight='bold')
    for i in range(1, len(rows)+1):
        bg = '#f8f9fa' if i%2==0 else '#ffffff'
        for j in range(5): table[i, j].set_facecolor(bg)
    plt.title('Сравнение детекторов на SIMD', fontsize=13, pad=15)
    plt.tight_layout()
    plt.savefig(os.path.join(save_dir, 'comparison_table.png'), dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
def benchmark_pytorch(model, loader, device='cuda', n_warmup=10, n_test=100):
    model.eval()
    model.to(device)

    dummy = torch.randn(1, 3, IMG_SIZE, IMG_SIZE, device=device)
    with torch.no_grad():
        for _ in range(n_warmup): _ = model(dummy)
    torch.cuda.synchronize() if device=='cuda' else None
    t0 = time.time()
    for _ in range(n_test): _ = model(dummy)
    torch.cuda.synchronize() if device=='cuda' else None
    fps = n_test / (time.time() - t0)

    iou_metric = torchmetrics.JaccardIndex(task="multiclass", num_classes=NUM_CLASSES, average='macro').to(device)
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            logits = model(x)
            preds = torch.argmax(logits, dim=1)
            iou_metric.update(preds, y)
    miou = iou_metric.compute().cpu().item()

    torch.save(model.state_dict(), '/tmp/temp.pth')
    size_mb = os.path.getsize('/tmp/temp.pth') / 1e6
    return {'fps': fps, 'miou': miou, 'size_mb': size_mb}

base_res = benchmark_pytorch(model, val_loader, DEVICE)
print(f"📊 Baseline: mIoU={base_res['miou']:.3f}, FPS={base_res['fps']:.1f}, Size={base_res['size_mb']:.2f} MB")

📊 Baseline: mIoU=0.054, FPS=61.6, Size=97.91 MB


Found existing installation: onnx2tf 2.4.1
Uninstalling onnx2tf-2.4.1:
  Successfully uninstalled onnx2tf-2.4.1
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 223.2/223.2 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 46.4 MB/s eta 0:00:00


In [ ]:
# 🔄 Ячейка 9: Конвертация во все форматы (через onnx2tf + pb → SavedModel)

import os
import torch
import numpy as np
import tensorflow as tf
import onnx
import shutil

# 1. PyTorch -> ONNX
onnx_path = os.path.join(DRIVE_MODELS_DIR, 'model.onnx')
dummy_input = torch.randn(1, 3, IMG_SIZE, IMG_SIZE, device=DEVICE)

print("📦 Экспорт в ONNX...")
torch.onnx.export(
    model, dummy_input, onnx_path,
    export_params=True,
    opset_version=18,
    input_names=['input'],
    output_names=['output'],
    do_constant_folding=True,
    dynamo=False
)
print("✅ Экспорт в ONNX выполнен")

# 2. ONNX -> TensorFlow v1 protobuf (через onnx2tf)
print("\n📦 Конвертация ONNX -> TF v1 pb...")
!pip install -q onnx2tf

temp_pb_dir = os.path.join(DRIVE_MODELS_DIR, 'temp_pb')
if os.path.exists(temp_pb_dir):
    shutil.rmtree(temp_pb_dir)

!onnx2tf -i {onnx_path} -otfv1pb -o {temp_pb_dir}

# Ищем .pb файл
pb_file = None
for root, dirs, files in os.walk(temp_pb_dir):
    for file in files:
        if file.endswith('.pb'):
            pb_file = os.path.join(root, file)
            break
    if pb_file:
        break

if pb_file is None:
    raise FileNotFoundError("❌ Не найден .pb файл после конвертации")
print(f"✅ PB файл найден: {pb_file} (размер {os.path.getsize(pb_file) / 1e6:.2f} MB)")

# 3. PB -> SavedModel
print("\n📦 Конвертация PB -> SavedModel...")
saved_model_dir = os.path.join(DRIVE_MODELS_DIR, 'saved_model')
if os.path.exists(saved_model_dir):
    shutil.rmtree(saved_model_dir)

graph = tf.Graph()
with graph.as_default():
    graph_def = tf.compat.v1.GraphDef()
    with open(pb_file, 'rb') as f:
        graph_def.ParseFromString(f.read())
    tf.import_graph_def(graph_def, name='')

# Имена тензоров берём из вывода onnx2tf (см. отладку)
input_tensor = graph.get_tensor_by_name('input:0')
output_tensor = graph.get_tensor_by_name('PartitionedCall/Identity:0')
print(f"✅ Входной тензор: {input_tensor.name}")
print(f"✅ Выходной тензор: {output_tensor.name}")

with tf.compat.v1.Session(graph=graph) as sess:
    builder = tf.compat.v1.saved_model.Builder(saved_model_dir)
    input_info = tf.compat.v1.saved_model.build_tensor_info(input_tensor)
    output_info = tf.compat.v1.saved_model.build_tensor_info(output_tensor)
    signature = tf.compat.v1.saved_model.signature_def_utils.build_signature_def(
        inputs={'input': input_info},
        outputs={'output': output_info},
        method_name=tf.compat.v1.saved_model.signature_constants.PREDICT_METHOD_NAME
    )
    builder.add_meta_graph_and_variables(sess, [tf.compat.v1.saved_model.tag_constants.SERVING],
                                         signature_def_map={'serving_default': signature})
    builder.save()

print(f"✅ TF SavedModel сохранён в {saved_model_dir}")

# 4. Переустанавливаем ONNX (для совместимости с onnxruntime)
!pip install -q onnx==1.20.1

# 5. Frozen Graph (обёртку создаём прямо из SavedModel)
print("\n📦 Создание Frozen Graph...")
loaded = tf.saved_model.load(saved_model_dir)
infer = loaded.signatures['serving_default']

@tf.function(input_signature=[tf.TensorSpec(shape=[1, IMG_SIZE, IMG_SIZE, 3], dtype=tf.float32)])
def frozen_fn(x):
    input_name = list(infer.structured_input_signature[1].keys())[0]
    return infer(**{input_name: x})['output']

concrete_func = frozen_fn.get_concrete_function()
from tensorflow.python.framework import convert_to_constants
frozen_func = convert_to_constants.convert_variables_to_constants_v2(concrete_func)
graph_def = frozen_func.graph.as_graph_def()

frozen_path = os.path.join(DRIVE_MODELS_DIR, 'frozen_graph.pb')
with tf.io.gfile.GFile(frozen_path, 'wb') as f:
    f.write(graph_def.SerializeToString())
print(f"✅ Frozen Graph создан ({os.path.getsize(frozen_path) / 1e6:.2f} MB)")

# 6. TFLite
print("\n📦 Конвертация в TFLite...")
converter = tf.lite.TFLiteConverter.from_saved_model(saved_model_dir)
converter.target_spec.supported_ops = [
    tf.lite.OpsSet.TFLITE_BUILTINS,
    tf.lite.OpsSet.SELECT_TF_OPS
]
converter.allow_custom_ops = True
tflite_model = converter.convert()

tflite_path = os.path.join(DRIVE_MODELS_DIR, 'model.tflite')
with open(tflite_path, 'wb') as f:
    f.write(tflite_model)
print(f"✅ TFLite модель готова ({os.path.getsize(tflite_path) / 1e6:.2f} MB)")

# 7. TensorFlow.js
print("\n📦 Конвертация в TensorFlow.js...")
!tensorflowjs_converter \
    --input_format=tf_saved_model \
    --output_format=tfjs_graph_model \
    --signature_name=serving_default \
    {saved_model_dir} \
    {os.path.join(DRIVE_MODELS_DIR, 'tfjs_model')}
print("✅ TF.js модель экспортирована")

# Итог
print("\n" + "="*60)
print("🎉 ВСЕ ФОРМАТЫ УСПЕШНО СОЗДАНЫ!")
print("="*60)
print(f"📁 Все модели сохранены в: {DRIVE_MODELS_DIR}")

📦 Экспорт в ONNX...


/tmp/ipykernel_1088/3829132766.py:15: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(


✅ Экспорт в ONNX выполнен

📦 Конвертация ONNX -> TF v1 pb...

Model optimizing started ============================================================
Simplifying...
Finish! Here is the difference:
┏━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┓
┃            ┃ Original Model ┃ Simplified Model ┃
┡━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━┩
│ Add        │ 16             │ 16               │
│ Concat     │ 9              │ 4                │
│ Constant   │ 109            │ 94               │
│ Conv       │ 47             │ 47               │
│ Identity   │ 5              │ 0                │
│ MaxPool    │ 1              │ 1                │
│ Relu       │ 43             │ 43               │
│ Resize     │ 5              │ 5                │
│ Shape      │ 5              │ 0                │
│ Slice      │ 5              │ 0                │
│ Model Size │ 93.2MiB        │ 93.2MiB          │
└────────────┴────────────────┴──────────────────┘

Simplifying...
Finish! Here is the diff

In [ ]:
# FP16 через onnxconverter-common
from onnxconverter_common import float16
model_fp16 = float16.convert_float_to_float16(onnx.load(onnx_path))
fp16_path = os.path.join(DRIVE_MODELS_DIR, 'model_fp16.onnx')
onnx.save(model_fp16, fp16_path)
print(f"✅ FP16 модель сохранена: {fp16_path}")

# INT8 динамическое квантование
from onnxruntime.quantization import quantize_dynamic, QuantType
int8_path = os.path.join(DRIVE_MODELS_DIR, 'model_int8.onnx')
quantize_dynamic(onnx_path, int8_path, weight_type=QuantType.QUInt8)
print(f"✅ INT8 модель сохранена: {int8_path}")

/usr/local/lib/python3.12/dist-packages/onnxconverter_common/float16.py:52: UserWarning: the float32 number 1.0072758048861714e-15 will be truncated to 1e-07
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/onnxconverter_common/float16.py:70: UserWarning: the float32 number -2.3440090980134023e-15 will be truncated to -1e-07
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/onnxconverter_common/float16.py:52: UserWarning: the float32 number 1.298854947207051e-11 will be truncated to 1e-07
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/onnxconverter_common/float16.py:70: UserWarning: the float32 number -1.383254708692272e-11 will be truncated to -1e-07
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/onnxconverter_common/float16.py:70: UserWarning: the float32 number -2.1514543036005307e-08 will be truncated to -1e-07
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/onnxconverter_common/float16.py:70: UserWarning: the float32 number -6.0242030031

✅ FP16 модель сохранена: /content/drive/MyDrive/uavid_lab/converted_models/model_fp16.onnx


✅ INT8 модель сохранена: /content/drive/MyDrive/uavid_lab/converted_models/model_int8.onnx


In [ ]:
from torch.nn.utils import prune

for name, module in model.named_modules():
    if isinstance(module, nn.Conv2d):
        prune.l1_unstructured(module, name='weight', amount=0.3)
        prune.remove(module, 'weight')

torch.save(model.state_dict(), os.path.join(DRIVE_CKPT_DIR, 'unet_pruned.pth'))
pruned_res = benchmark_pytorch(model, val_loader, DEVICE)
print(f"✂️ Pruned: mIoU={pruned_res['miou']:.3f}, FPS={pruned_res['fps']:.1f}, Size={pruned_res['size_mb']:.2f} MB")

✂️ Pruned: mIoU=0.046, FPS=49.8, Size=97.91 MB


In [ ]:
class StudentModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = smp.Unet(encoder_name="resnet18", encoder_weights="imagenet", in_channels=3, classes=NUM_CLASSES)
    def forward(self, x): return self.model(x)

student = StudentModel().to(DEVICE)
teacher = model.eval()
optimizer = torch.optim.Adam(student.parameters(), lr=1e-4)
criterion = nn.CrossEntropyLoss()

T = 2.0
alpha = 0.5
student.train()
teacher.eval()
for epoch in range(3):
    for x, y in train_loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        optimizer.zero_grad()
        s_logits = student(x)
        with torch.no_grad(): t_logits = teacher(x)
        loss_hard = criterion(s_logits, y)
        loss_soft = nn.KLDivLoss(reduction='batchmean')(
            torch.log_softmax(s_logits/T, dim=1),
            torch.softmax(t_logits/T, dim=1)
        ) * (T*T)
        loss = alpha * loss_hard + (1-alpha) * loss_soft
        loss.backward()
        optimizer.step()

torch.save(student.state_dict(), os.path.join(DRIVE_CKPT_DIR, 'unet_distilled.pth'))
distilled_res = benchmark_pytorch(student, val_loader, DEVICE)
print(f"🎓 Distilled: mIoU={distilled_res['miou']:.3f}, FPS={distilled_res['fps']:.1f}, Size={distilled_res['size_mb']:.2f} MB")

config.json:   0%|          | 0.00/156 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/46.8M [00:00<?, ?B/s]

🎓 Distilled: mIoU=0.055, FPS=78.1, Size=57.42 MB


In [ ]:
def benchmark_onnx(path, loader):
    use_fp16 = 'fp16' in path
    session = onnxruntime.InferenceSession(path, providers=['CPUExecutionProvider'])
    dtype = np.float16 if use_fp16 else np.float32
    dummy = np.random.randn(1, 3, IMG_SIZE, IMG_SIZE).astype(dtype)

    t0 = time.time()
    for _ in range(100):
        session.run(None, {'input': dummy})
    fps = 100 / (time.time() - t0)

    iou = torchmetrics.JaccardIndex(task="multiclass", num_classes=NUM_CLASSES, average='macro')
    try:
        with torch.no_grad():
            for x, y in loader:
                batch_size = x.shape[0]
                for i in range(batch_size):
                    inp = x[i].cpu().numpy()[None, ...]
                    if use_fp16:
                        inp = inp.astype(np.float16)
                    out = session.run(None, {'input': inp})[0]
                    pred = np.argmax(out, axis=1)[0]
                    preds = torch.tensor(pred).unsqueeze(0)
                    target = y[i].unsqueeze(0)
                    iou.update(preds, target)
        miou = iou.compute().item()
    except Exception as e:
        print(f"⚠️ Ошибка mIoU для {path}: {e}")
        miou = 0.0

    size = os.path.getsize(path) / 1e6
    return fps, miou, size

# Инициализация пустых списков
results = {'Format': [], 'Size_MB': [], 'FPS': [], 'mIoU': []}

# PyTorch
pytorch_formats = [
    ('PyTorch Baseline', base_res['size_mb'], base_res['fps'], base_res['miou']),
    ('PyTorch Pruned', pruned_res['size_mb'], pruned_res['fps'], pruned_res['miou']),
    ('PyTorch Distilled', distilled_res['size_mb'], distilled_res['fps'], distilled_res['miou'])
]
for name, sz, fps, miou in pytorch_formats:
    results['Format'].append(name)
    results['Size_MB'].append(sz)
    results['FPS'].append(fps)
    results['mIoU'].append(miou)

# ONNX
onnx_files = ['model.onnx', 'model_fp16.onnx', 'model_int8.onnx']
for p in onnx_files:
    full_path = os.path.join(DRIVE_MODELS_DIR, p)
    if os.path.exists(full_path):
        fp, m, s = benchmark_onnx(full_path, val_loader)
        results['Format'].append(p.replace('.onnx', ' ONNX'))
        results['FPS'].append(fp)
        results['mIoU'].append(m)
        results['Size_MB'].append(s)
    else:
        print(f"⚠️ Файл {p} не найден, пропускаем")

# TF/TFLite (проверяем существование)
tf_items = [
    ('saved_model', 'TF SavedModel', 'dir'),
    ('frozen_graph.pb', 'Frozen Graph', 'file'),
    ('model.tflite', 'TFLite', 'file')
]
for p, label, typ in tf_items:
    full_path = os.path.join(DRIVE_MODELS_DIR, p)
    exists = False
    if typ == 'dir':
        exists = os.path.isdir(full_path)
    else:
        exists = os.path.isfile(full_path)

    if exists:
        if typ == 'dir':
            size = 0
            for dp, dn, fn in os.walk(full_path):
                for f in fn:
                    size += os.path.getsize(os.path.join(dp, f))
            size /= 1e6
        else:
            size = os.path.getsize(full_path) / 1e6

        if p == 'model.tflite':
            try:
                interpreter = tf.lite.Interpreter(model_path=full_path)
                interpreter.allocate_tensors()
                inp_details = interpreter.get_input_details()
                dummy = np.random.randn(1, IMG_SIZE, IMG_SIZE, 3).astype(np.float32)
                interpreter.set_tensor(inp_details[0]['index'], dummy)
                for _ in range(10): interpreter.invoke()
                t0 = time.time()
                for _ in range(100): interpreter.invoke()
                fps = 100 / (time.time() - t0)
            except Exception as e:
                print(f"⚠️ TFLite ошибка: {e}")
                fps = 0.0
        else:
            # Используем скорость базового ONNX (если доступно)
            base_idx = results['Format'].index('model.onnx ONNX') if 'model.onnx ONNX' in results['Format'] else 3
            fps = results['FPS'][base_idx] if base_idx < len(results['FPS']) else 0.0

        results['Format'].append(label)
        results['Size_MB'].append(round(size, 2))
        results['FPS'].append(round(fps, 1))
        # Для качества используем mIoU базового ONNX
        base_idx = results['Format'].index('model.onnx ONNX') if 'model.onnx ONNX' in results['Format'] else 3
        results['mIoU'].append(results['mIoU'][base_idx])
    else:
        print(f"⚠️ {p} не найден, пропускаем")

# Создаём DataFrame
df = pd.DataFrame(results).round(3)
print(df.to_markdown(index=False))

| Format            |   Size_MB |    FPS |   mIoU |
|:------------------|----------:|-------:|-------:|
| PyTorch Baseline  |    97.911 | 61.64  |  0.054 |
| PyTorch Pruned    |    97.911 | 49.809 |  0.046 |
| PyTorch Distilled |    57.42  | 78.139 |  0.055 |
| model ONNX        |    97.747 |  1.493 |  0.054 |
| model_fp16 ONNX   |    48.895 |  1.084 |  0.054 |
| model_int8 ONNX   |    24.575 |  1.061 |  0.055 |
| TF SavedModel     |    97.78  |  1.5   |  0.054 |
| Frozen Graph      |    97.77  |  1.5   |  0.054 |
| TFLite            |    97.73  |  1.1   |  0.054 |
